In [ ]:
import pandas as pd

# 1. Cargar los datos limpios creados en la fase anterior
ruta_accesos = "../../datos/procesados/accesos_historico_total.csv"
ruta_socios = "../../datos/procesados/socios_historico_total.csv"

df_accesos = pd.read_csv(ruta_accesos)
df_socios = pd.read_csv(ruta_socios)

# 2. Conversión de tiempo
df_accesos['fecha'] = pd.to_datetime(df_accesos['fecha'])

fecha_referencia = df_accesos['fecha'].max()

# --- FASE 1: FEATURE ENGINEERING ---

# Calculamos Recencia y Frecuencia agrupando por socio
comportamiento = df_accesos.groupby('id_socio').agg(
    ultima_visita=('fecha', 'max'),       # La fecha más reciente que vino
    total_visitas=('fecha', 'count')      # Cuántas veces ha venido en total
).reset_index()

# Calculamos los días exactos desde su última visita hasta el final de los datos
comportamiento['dias_desde_ultima_visita'] = (fecha_referencia - comportamiento['ultima_visita']).dt.days

# Calculamos la Estacionalidad (Día favorito de la semana para entrenar)
df_accesos['dia_semana'] = df_accesos['fecha'].dt.dayofweek
dia_favorito = df_accesos.groupby('id_socio')['dia_semana'].apply(lambda x: x.mode()[0]).reset_index()
dia_favorito.rename(columns={'dia_semana': 'dia_favorito'}, inplace=True)

# Unimos las métricas de comportamiento
comportamiento = pd.merge(comportamiento, dia_favorito, on='id_socio', how='left')

# --- FASE 2: EL CRUCE FINAL ---

# Unimos el maestro de socios (izq) con su comportamiento calculado (der)
# Usamos how='left' para no perder a los socios que se apuntaron pero nunca fueron
df_abt = pd.merge(df_socios, comportamiento, on='id_socio', how='left')

# --- FASE 3: TRATAMIENTO DE NULOS ---

df_abt['total_visitas'] = df_abt['total_visitas'].fillna(0)

df_abt['dias_desde_ultima_visita'] = df_abt['dias_desde_ultima_visita'].fillna(999) 
df_abt['dia_favorito'] = df_abt['dia_favorito'].fillna(-1) # -1 indica que no tiene día favorito porque nunca ha venido

# Tiramos la fecha de última visita porque los algoritmos prefieren el número de días
df_abt = df_abt.drop(columns=['ultima_visita'])

print(f"Cruce completado con éxito")
print(f"Total de socios en la tabla final: {len(df_abt)}")


¡Cruce completado con éxito!
Total de socios en la tabla final: 8658


In [5]:
# --- FASE FINAL: GUARDADO ---
ruta_guardado = "../../datos/procesados"
ruta_abt = f"{ruta_guardado}/abt_socios_modelo.csv"
df_abt.to_csv(ruta_abt, index=False)

print(f"Archivo matriz listo para el modelo: {ruta_abt}")

Archivo matriz listo para el modelo: ../../datos/procesados/abt_socios_modelo.csv
